In [2]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

In [3]:
# -------------------------------------------------------------------------
# CONFIGURACIÓN: LISTA DE HOTELES
# -------------------------------------------------------------------------
hoteles = {
    "Vitoria": 'https://www.booking.com/reviews/es/hotel/libere-vitoria-centro.es.html',
    "Donosti": 'https://www.booking.com/reviews/es/hotel/koisi-hostel.es.html',
    "BilbaoMuseo": 'https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-guggenheim.es.html',
    "BilbaoLaVieja": 'https://www.booking.com/reviews/es/hotel/apartamentos-libere-bilbao-la-vieja.es.html',
    "ValenciaAbastos": 'https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-abastos.es.html',
    "PamplonaYamaguchi": 'https://www.booking.com/reviews/es/hotel/apartamentos-libere-pamplona-yamaguchi.es.html',
    "ValenciaJardinBotanico": 'https://www.booking.com/reviews/es/hotel/apartamentos-libere-valencia-jardin-botanico.es.html',
    "MadridPalacioReal": 'https://www.booking.com/reviews/es/hotel/libere-madrid-palacio-real.es.html',
    "MalagaTeatroRomano": 'https://www.booking.com/reviews/es/hotel/apartamentosliberemalagateatroromano.es.html',
    "GranadaCatedral": 'https://www.booking.com/reviews/es/hotel/apartamentos-libere-granada-catedral.es.html',
    "MalagaLaMerced": 'https://www.booking.com/reviews/es/hotel/libere-malaga-la-merced.es.html',
    "CordobaPatio": 'https://www.booking.com/reviews/es/hotel/libere-cordoba-patio-santa-marta.es.html'
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9",
}

# -------------------------------------------------------------------------
# FUNCIÓN DE EXTRACCIÓN
# -------------------------------------------------------------------------
def extraer_reviews_hotel(nombre_hotel, url_base):
    todas_las_reviews_hotel = []
    pagina_actual = 1
    
    params = {
        'customer_type': 'total',
        'hp_nav': '0',
        'keep_landing': '1',
        'order': 'featuredreviews',
        'r_lang': 'es',
        'rows': '75',
    }

    print(f"\n>>> INICIANDO EXTRACCIÓN: {nombre_hotel}")
    
    while True:
        print(f"   Página {pagina_actual}...", end="\r")
        params['page'] = pagina_actual
        
        try:
            response = requests.get(url_base, headers=headers, params=params)
            
            if response.status_code != 200:
                print(f"\n   Error {response.status_code} en {nombre_hotel}. Saltando...")
                break
            
            soup = BeautifulSoup(response.text, 'html.parser')
            lista_reviews = soup.find_all("li", class_="review_item")
            
            if not lista_reviews:
                print(f"\n   Fin de reseñas para {nombre_hotel}.")
                break
            
            for review in lista_reviews:
                item = {'hotel': nombre_hotel}
                
                fecha_tag = review.find("p", class_="review_item_date")
                item['fecha'] = fecha_tag.get_text(strip=True).replace("Comentario enviado el ", "") if fecha_tag else "Sin fecha"
                
                score_tag = review.find("div", class_="review_item_review_score")
                item['puntuacion'] = score_tag.get_text(strip=True) if score_tag else "Sin puntuación"
                
                name_tag = review.find("p", class_="reviewer_name")
                item['nombre'] = name_tag.get_text(strip=True) if name_tag else "Anónimo"
                
                title_tag = review.find("div", class_="review_item_header_content")
                item['titulo'] = title_tag.get_text(strip=True).strip('"') if title_tag else ""
                
                pos_tag = review.find("p", class_="review_pos")
                item['positivo'] = pos_tag.get_text(strip=True).replace("눇", "") if pos_tag else ""

                neg_tag = review.find("p", class_="review_neg")
                item['negativo'] = neg_tag.get_text(strip=True).replace("눉", "") if neg_tag else ""
                
                todas_las_reviews_hotel.append(item)
            
            pagina_actual += 1
            time.sleep(1.5)
            
        except Exception as e:
            print(f"\n   Error crítico en {nombre_hotel}: {e}")
            break
            
    return todas_las_reviews_hotel

# -------------------------------------------------------------------------
# EJECUCIÓN PRINCIPAL
# -------------------------------------------------------------------------

resultados_totales = []

for ciudad, url in hoteles.items():
    datos_hotel = extraer_reviews_hotel(ciudad, url)
    if datos_hotel:
        resultados_totales.extend(datos_hotel)
    else:
        print(f"   Advertencia: No se obtuvieron datos para {ciudad}")

if resultados_totales:
    df = pd.DataFrame(resultados_totales)
    
    nombre_archivo = 'Datos/NLP/reviews_booking.csv'
    
    df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    
    print(f"\n" + "="*40)
    print(f"PROCESO COMPLETADO")
    print(f"Total de reseñas extraídas: {len(df)}")
    print(f"Archivo guardado como: {nombre_archivo}")
    print("="*40)
else:
    print("\nNo se extrajeron datos de ningún hotel.")


>>> INICIANDO EXTRACCIÓN: Vitoria
   Página 43...
   Fin de reseñas para Vitoria.

>>> INICIANDO EXTRACCIÓN: Donosti
   Página 18...
   Fin de reseñas para Donosti.

>>> INICIANDO EXTRACCIÓN: BilbaoMuseo
   Página 5...
   Fin de reseñas para BilbaoMuseo.

>>> INICIANDO EXTRACCIÓN: BilbaoLaVieja
   Página 7...
   Fin de reseñas para BilbaoLaVieja.

>>> INICIANDO EXTRACCIÓN: ValenciaAbastos
   Página 6...
   Fin de reseñas para ValenciaAbastos.

>>> INICIANDO EXTRACCIÓN: PamplonaYamaguchi
   Página 14...
   Fin de reseñas para PamplonaYamaguchi.

>>> INICIANDO EXTRACCIÓN: ValenciaJardinBotanico
   Página 5...
   Fin de reseñas para ValenciaJardinBotanico.

>>> INICIANDO EXTRACCIÓN: MadridPalacioReal
   Página 6...
   Fin de reseñas para MadridPalacioReal.

>>> INICIANDO EXTRACCIÓN: MalagaTeatroRomano
   Página 4...
   Fin de reseñas para MalagaTeatroRomano.

>>> INICIANDO EXTRACCIÓN: GranadaCatedral
   Página 7...
   Fin de reseñas para GranadaCatedral.

>>> INICIANDO EXTRACCIÓN: Malaga